In [3]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.spatial.distance import squareform

In [4]:
# Parameters
n_reps = 2+3  # Number of representatives per cluster
diversity_weight = 0.3  # 0 = only centrality, 1 = only diversity

# Paths
cluster_path = "../data/final_clusters/spectral_13_clusters.csv"
similarity_path= "../data/similarities/anime_similarity_upper_triangle.csv"
result_path = f"../data/results/spectral_13_clusters_{n_reps}_reps.csv"
anime_path = "../data/anime.csv"



# Load data
similarity_df = pd.read_csv(similarity_path)
cluster_df = pd.read_csv(cluster_path)
cluster_df.set_index("anime_id", inplace=True)

# Convert edge list to distance matrix
all_anime_ids = sorted(set(similarity_df['anime_id_1'].tolist() + similarity_df['anime_id_2'].tolist()))
anime_to_idx = {anime_id: i for i, anime_id in enumerate(all_anime_ids)}

# Build similarity matrix from edge list
n_anime = len(all_anime_ids)
sim_matrix = np.zeros((n_anime, n_anime))
for _, row in similarity_df.iterrows():
    i, j = anime_to_idx[row['anime_id_1']], anime_to_idx[row['anime_id_2']]
    sim_matrix[i, j] = sim_matrix[j, i] = row['similarity']

# Convert to distance matrix
distance_matrix = 1 - sim_matrix
distance_df = pd.DataFrame(distance_matrix, index=all_anime_ids, columns=all_anime_ids)

# Find n representatives for each cluster
all_reps = []
for cluster_id in cluster_df["cluster"].unique():
    cluster_anime = cluster_df[cluster_df["cluster"] == cluster_id].index
    
    # Only use anime that exist in our similarity data
    cluster_anime = [anime_id for anime_id in cluster_anime if anime_id in distance_df.index]
    
    if len(cluster_anime) <= n_reps:
        # If cluster smaller than n, take all
        all_reps.extend([(cluster_id, i+1, anime_id) for i, anime_id in enumerate(cluster_anime)])
        continue
    
    cluster_distances = distance_df.loc[cluster_anime, cluster_anime]
    sum_distances = cluster_distances.sum(axis=1)
    
    # Find diverse representatives
    reps = [sum_distances.idxmin()]  # Start with most central
    remaining = cluster_distances.index.drop(reps[0])
    
    for _ in range(n_reps - 1):
        scores = []
        for candidate in remaining:
            centrality = sum_distances[candidate] / sum_distances.max()
            diversity = min([cluster_distances.loc[candidate, rep] for rep in reps]) / cluster_distances.max().max()
            scores.append((1-diversity_weight) * centrality + diversity_weight * (1-diversity))
        
        best_idx = np.argmin(scores)
        best_candidate = remaining[best_idx]
        reps.append(best_candidate)
        remaining = remaining.drop(best_candidate)
    
    all_reps.extend([(cluster_id, i+1, rep) for i, rep in enumerate(reps)])

# Create results dataframe
results_df = pd.DataFrame(all_reps, columns=['cluster_id', 'rank', 'anime_id'])
results_df.set_index('anime_id', inplace=True)

# Add anime metadata from your existing anime.csv
anime_df = pd.read_csv(anime_path, index_col='anime_id')  # Adjust path as needed
final_df = results_df.join(anime_df, how='left')

# Keep only desired columns
columns_to_keep = ['cluster_id', 'title', 'main_pic']  # Adjust column names as needed
final_df = final_df[columns_to_keep]
final_df.to_csv(result_path)

print(f"Found {len(results_df)} representatives across {cluster_df['cluster'].nunique()} clusters")
print(final_df.head(10))

Found 65 representatives across 13 clusters
          cluster_id                                              title   
anime_id                                                                  
28851              6                                     Koe no Katachi  \
11757              6                                   Sword Art Online   
29803              6                                           Overlord   
38408              6                   Boku no Hero Academia 4th Season   
36949              6  Shokugeki no Souma: San no Sara - Tootsuki Res...   
43                 2                                   Koukaku Kidoutai   
585                2                                   Mimi wo Sumaseba   
486                2                  Kino no Tabi: The Beautiful World   
5681               2                                        Summer Wars   
759                2                                   Tokyo Godfathers   

                                                   main